# Exercise 6: Dimensionality Reduction 2 — LFP Blind Source Separation & Nonlinear Methods

| | |
|---|---|
| **Topic** | FastICA on multichannel LFP, theta-triggered analysis, bandpass effects, denoising, nonlinear methods (Isomap, t-SNE) |
| **Data** | `lfp.mat` (multichannel LFP from linear probe) + `spike_waveshapes.mat` (for nonlinear bonus) |
| **Prereqs** | Exercise 5 (DimRed 1); Anton's lecture on BSS and nonlinear methods |
| **Time** | ~2 hours |

## Learning objectives

- Apply FastICA to multichannel LFP
- Interpret spatial profiles of mixing matrix columns
- Perform theta-triggered averaging on IC activations
- Explore frequency-band effects on ICA components
- Use ICA for denoising and artifact removal
- Compare PCA with nonlinear methods (Isomap, t-SNE)


## Setup

In [ ]:
import numpy as np
import scipy.io as sio
from scipy.stats import zscore
from scipy import signal
from scipy.signal import find_peaks
import matplotlib.pyplot as plt
from itertools import combinations

from sklearn.decomposition import PCA, FastICA
from sklearn.mixture import GaussianMixture
from sklearn.manifold import Isomap, TSNE

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

## Part 2: LFP Blind Source Separation

Apply FastICA to multichannel local field potentials (LFP) recorded from a
linear silicon probe. Sampling rate: 1250 Hz.

In [ ]:
# Load LFP data (provided)
lfp_data = sio.loadmat('../../data/spectral/lfp.mat')
lfp = lfp_data['lfp'].astype(np.float64)
lfp = zscore(lfp, axis=0)
fs = 1250
n_samples_lfp, n_ch_lfp = lfp.shape
print(f"LFP: {n_samples_lfp} samples, {n_ch_lfp} channels, "
      f"{n_samples_lfp / fs:.1f} s, fs={fs} Hz")

In [ ]:
# Helper functions (provided -- use these in your solutions below)

def butter_bandpass_filter(data, lowcut, highcut, fs, order=4):
    """Apply zero-phase Butterworth bandpass filter."""
    nyq = 0.5 * fs
    b, a = signal.butter(order, [lowcut / nyq, highcut / nyq], btype='band')
    return signal.filtfilt(b, a, data, axis=0)


def triggered_average(x, triggers, half_win):
    """Average signal x in windows of +/-half_win around each trigger."""
    segs = []
    for t in triggers:
        if t - half_win >= 0 and t + half_win <= len(x):
            segs.append(x[t - half_win:t + half_win])
    return np.mean(segs, axis=0) if segs else np.zeros(2 * half_win)

### Task 2a: FastICA -- Spatial Profiles and CSD

Apply FastICA to the raw LFP. Plot spatial profiles of the mixing matrix columns
and compute the current source density (CSD) from the smoothed mixing matrix.

**Think about it:** The mixing matrix A maps IC activations to channels:
LFP = S . A^T. What does each column of A represent physically?

In [ ]:
# TODO: FastICA on LFP
# 1. Fit FastICA with n_components=16
# 2. Extract mixing matrix A = ica.mixing_
# 3. Smooth A spatially (lowpass filter over channels) and compute CSD = -d^2 A/dx^2
# 4. Plot A, smoothed A, and CSD as images
# 5. Plot individual spatial profiles (mixing matrix columns) as depth plots

raise NotImplementedError("Your code here")

<details>
<summary><strong>Hint 1: Spatial profiles</strong></summary>

Each column of the mixing matrix A is the spatial "fingerprint" of that IC across
channels (depth). A dipolar profile (positive at one depth, negative at another)
suggests a localized neural source. The CSD (-d^2 A/dx^2) reveals current
sinks and sources.
</details>

<details>
<summary><strong>Hint 2: Computing CSD</strong></summary>

```python
# Smooth mixing matrix over channels (axis=0)
b, a = signal.butter(2, 0.2, btype='low')
A_smooth = signal.filtfilt(b, a, A, axis=0)
CSD_A = -np.diff(A_smooth, n=2, axis=0)
```
</details>

In [ ]:
# --- Checkpoint 2a: LFP FastICA ---
# Verifies your Task 2a output. Expects `S` (IC activations,
# shape n_samples_lfp x n_comp) and `A` (mixing matrix, n_ch_lfp x n_comp).
try:
    assert S.shape == (n_samples_lfp, n_comp), \
        f'S should be ({n_samples_lfp}, {n_comp}), got {S.shape}'
    assert A.shape == (n_ch_lfp, n_comp), \
        f'A should be ({n_ch_lfp}, {n_comp}), got {A.shape}'
    print(f'\u2713 Checkpoint 2a passed: S {S.shape}, A {A.shape}')
except NameError as e:
    print(f'\u26a0 {e} \u2014 finish Task 2a then re-run this cell')

### Task 2b: IC Activation Timeseries

Plot the first few seconds of each IC's activation timeseries. Identify
components with oscillatory vs noisy behavior.

In [ ]:
# TODO: Plot IC activations (e.g., first 3 seconds)

raise NotImplementedError("Your code here")

### Task 2c: Theta-Triggered Averaging

Filter channel 0 in the theta band (5-12 Hz), detect local minima (troughs),
and compute the average theta cycle shape using each IC's activations.

**Think about it:** Which ICs should show strong theta modulation?
How does this relate to their spatial profiles?

In [ ]:
# TODO: Theta-triggered averaging
# 1. Bandpass filter channel 0 at 5-12 Hz using butter_bandpass_filter
# 2. Detect troughs using find_peaks on the negated signal
# 3. Compute triggered averages of each IC in +/-70 ms windows
# 4. Plot the triggered averages

raise NotImplementedError("Your code here")

<details>
<summary><strong>Hint 1: Theta trough detection</strong></summary>

Theta troughs are local minima. Detect them with `find_peaks(-theta)`.
Set `distance=int(fs / 12)` to avoid detecting multiple peaks within one cycle.
</details>

<details>
<summary><strong>Hint 2: Using the helpers</strong></summary>

```python
theta = butter_bandpass_filter(lfp[:, 0], 5, 12, fs)
troughs, _ = find_peaks(-theta, distance=int(fs / 12))
half_win = int(0.070 * fs)  # +/- 70 ms
avg = triggered_average(S[:, ic], troughs, half_win)
```
</details>

### Tasks 2d-f: Advanced ICA Analysis

**2d.** Explore the effect of bandpass filtering (5-30 Hz and 30-150 Hz) on ICA
performance. Apply ICA separately to each filtered signal and compare mixing matrices.

**2e.** Explore denoising by reconstructing LFP from fewer ICs (e.g., 4, 8, 12).
Compare the reconstruction to the original signal.

**2f.** Identify EMG/artifact components (spatially uniform loading, non-stationary
activations). Remove them and compare theta-triggered averages of original vs
cleaned LFP.

In [ ]:
# TODO: Task 2d -- Bandpass filtering effects on ICA
# Apply ICA separately to theta-beta (5-30 Hz) and gamma (30-150 Hz) filtered LFP
# Compare mixing matrices to the broadband result

raise NotImplementedError("Your code here")

In [ ]:
# TODO: Task 2e -- Denoising by compression
# Reconstruct LFP using only the first K ICs (try K=4, 8, 12)
# S_reduced = S.copy(); S_reduced[:, K:] = 0; lfp_denoised = S_reduced @ A.T

raise NotImplementedError("Your code here")

In [ ]:
# TODO: Task 2f -- EMG artifact removal
# 1. Identify artifact ICs: low spatial specificity (std/mean of |A| columns)
#    and/or high temporal nonstationarity (CV of windowed variance)
# 2. Zero out those ICs and reconstruct
# 3. Compare theta-triggered averages: original vs cleaned

raise NotImplementedError("Your code here")

## Bonus: Nonlinear Dimensionality Reduction

Compare PCA with Isomap and t-SNE on the spike waveform data.

**Think about it:** When would nonlinear methods outperform PCA?
Why might Isomap fail on some datasets?

> **Note:** `umap-learn` is not installed. Use `sklearn.manifold.Isomap` and
> `sklearn.manifold.TSNE` instead.

In [ ]:
# TODO: Nonlinear dimensionality reduction
# 1. Subsample ~5000 spikes for speed (Isomap/t-SNE are slow on large N)
# 2. Use first 10 PCA scores as input (from full-space PCA)
# 3. Fit Isomap (n_neighbors=15) and t-SNE (perplexity=30)
# 4. Compare PCA, Isomap, and t-SNE in side-by-side scatter plots

raise NotImplementedError("Your code here")

<details>
<summary><strong>Hint: Why Isomap can fail on disconnected graphs</strong></summary>

Isomap builds a k-nearest-neighbor graph and computes geodesic (shortest-path)
distances. If the graph has disconnected components -- some points can't reach
others -- the distances are undefined and the algorithm fails. Fix: increase
`n_neighbors` to connect the graph, or use t-SNE (which doesn't rely on graph
connectivity).
</details>

## Reflection

**Questions to consider:**
1. How many PCs are needed to capture most of the spike waveform variance?
2. Which channels were most informative for cluster separation? Why?
3. How did the GMM clustering compare to the provided labels?
4. What did the ICA spatial profiles reveal about theta generators in the LFP?
5. How did nonlinear methods compare to PCA for visualizing spike clusters?

## Reflection

**Questions to consider:**
1. What spatial profiles did ICA find in the LFP? Can you identify any as biologically meaningful (e.g. a laminar source, an artifact)?
2. How did bandpass filtering change the ICA decomposition?
3. Did ICA help denoise the LFP? Which ICs would you keep vs discard?
4. How do Isomap and t-SNE compare with PCA for visualising spike waveshape clusters?
